# Session 10: Network Availability and Monitoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/03-computer-networks/code/session-10-network-availability.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-10_syslog_monitor.py`

In [1]:
"""Session 10 - A tiny syslog collector on localhost plus a log analyser.

1. Starts a UDP syslog listener on 127.0.0.1 (random high port, not 514,
   so no admin rights are needed).
2. "Devices" send RFC 3164-style messages: <PRI>timestamp host message,
   where PRI = facility * 8 + severity.
3. The collector decodes facility/severity and summarises what needs attention.
Run:  python session-10_syslog_monitor.py
"""
import re
import socket
import threading
import time
from collections import Counter

SEVERITY = ["Emergency", "Alert", "Critical", "Error", "Warning", "Notice", "Informational", "Debug"]
FACILITY = {0: "kern", 1: "user", 4: "auth", 10: "authpriv", 16: "local0", 23: "local7"}

MESSAGES = [  # (facility, severity, host, text)
    (23, 5, "core-sw1", "%LINK-5-CHANGED: Interface Gi1/0/24, changed state to up"),
    (23, 3, "core-sw1", "%LINK-3-UPDOWN: Interface Gi1/0/7, changed state to down"),
    (4, 4, "fw1", "Deny tcp src outside:198.51.100.7/51022 dst inside:10.1.5.20/3389"),
    (4, 4, "fw1", "Deny tcp src outside:198.51.100.7/51023 dst inside:10.1.5.21/3389"),
    (10, 5, "web01", "sshd: Failed password for admin from 10.1.9.50"),
    (10, 5, "web01", "sshd: Failed password for admin from 10.1.9.50"),
    (10, 6, "web01", "sshd: Accepted publickey for deploy from 10.1.9.10"),
    (23, 2, "edge-r1", "%SYS-2-MALLOCFAIL: Memory allocation failed"),
    (16, 6, "app01", "health check OK in 42 ms"),
]

received = []


def collector(sock, expected):
    while len(received) < expected:
        data, _ = sock.recvfrom(2048)
        received.append(data.decode())


def parse(line):
    m = re.match(r"<(\d+)>(\w{3} +\d+ [\d:]+) (\S+) (.*)", line)
    pri = int(m.group(1))
    return {"facility": FACILITY.get(pri // 8, pri // 8), "severity": pri % 8,
            "time": m.group(2), "host": m.group(3), "msg": m.group(4)}


if __name__ == "__main__":
    srv = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    srv.bind(("127.0.0.1", 0))
    port = srv.getsockname()[1]
    th = threading.Thread(target=collector, args=(srv, len(MESSAGES)), daemon=True)
    th.start()

    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as dev:
        for i, (fac, sev, host, text) in enumerate(MESSAGES):
            line = f"<{fac * 8 + sev}>Oct 26 09:{10 + i:02d}:00 {host} {text}"
            dev.sendto(line.encode(), ("127.0.0.1", port))
            time.sleep(0.01)
    th.join(timeout=2)

    events = [parse(l) for l in received]
    print(f"Collected {len(events)} messages. First raw line: {received[0]}")
    print("\nBy severity:")
    for sev, n in sorted(Counter(e["severity"] for e in events).items()):
        print(f"  {sev} {SEVERITY[sev]:<14} {'#' * n} {n}")
    print("\nNeeds attention now (severity 0-3):")
    for e in events:
        if e["severity"] <= 3:
            print(f"  {e['time']} {e['host']:<8} {SEVERITY[e['severity']]:<9} {e['msg']}")
    fails = Counter(re.search(r"from (\S+)", e["msg"]).group(1) for e in events if "Failed password" in e["msg"])
    denies = Counter(re.search(r"outside:([\d.]+)", e["msg"]).group(1) for e in events if e["msg"].startswith("Deny"))
    print("\nSecurity patterns:", dict(fails), "failed SSH;", dict(denies), "blocked RDP attempts")

Collected 9 messages. First raw line: <189>Oct 26 09:10:00 core-sw1 %LINK-5-CHANGED: Interface Gi1/0/24, changed state to up

By severity:
  2 Critical       # 1
  3 Error          # 1
  4 Warning        ## 2
  5 Notice         ### 3
  6 Informational  ## 2

Needs attention now (severity 0-3):
  Oct 26 09:11:00 core-sw1 Error     %LINK-3-UPDOWN: Interface Gi1/0/7, changed state to down
  Oct 26 09:17:00 edge-r1  Critical  %SYS-2-MALLOCFAIL: Memory allocation failed

Security patterns: {'10.1.9.50': 2} failed SSH; {'198.51.100.7': 2} blocked RDP attempts


## `session-10_metrics.py`

In [2]:
"""Session 10 - Network performance metrics from ping-style samples:
latency, jitter, packet loss, throughput/utilisation and the meaning of "nines".

The RTT samples are a recorded lab capture (None = lost packet).
Run:  python session-10_metrics.py
"""
import statistics

SAMPLES = {
    "LAN to file server":  [0.6, 0.5, 0.7, 0.6, 0.5, 0.6, 0.8, 0.6, 0.5, 0.6],
    "Branch over VPN":     [38, 41, 39, 120, 40, 42, None, 39, 41, 40],
    "Congested Wi-Fi":     [12, 85, 30, None, 140, 22, None, 95, 18, None],
}


def metrics(rtts):
    got = [r for r in rtts if r is not None]
    loss = 100 * (len(rtts) - len(got)) / len(rtts)
    jitter = statistics.mean(abs(a - b) for a, b in zip(got, got[1:]))   # mean delay variation
    return statistics.mean(got), min(got), max(got), jitter, loss


def voip_ok(avg, jitter, loss):
    return avg < 150 and jitter < 30 and loss < 1


if __name__ == "__main__":
    print(f"{'Link':<20}{'avg ms':>8}{'min':>6}{'max':>6}{'jitter':>8}{'loss':>7}  VoIP?")
    for name, s in SAMPLES.items():
        avg, lo, hi, j, loss = metrics(s)
        print(f"{name:<20}{avg:>8.1f}{lo:>6}{hi:>6}{j:>8.1f}{loss:>6.0f}%  {'yes' if voip_ok(avg, j, loss) else 'no'}")

    print("\nUtilisation of a 1 Gbps uplink from interface counters (5-minute interval):")
    bytes_before, bytes_after, secs = 8_250_000_000, 39_750_000_000, 300
    bps = (bytes_after - bytes_before) * 8 / secs
    print(f"  {bps / 1e6:,.0f} Mbps = {bps / 1e9:.0%} utilisation (alert threshold 70%)")

    print("\nAvailability 'nines' -> allowed downtime per year:")
    for a in (99, 99.9, 99.99, 99.999):
        mins = (1 - a / 100) * 365 * 24 * 60
        print(f"  {a:>7}%  {mins / 60:>7.2f} hours  ({mins:,.1f} minutes)")

Link                  avg ms   min   max  jitter   loss  VoIP?
LAN to file server       0.6   0.5   0.8     0.1     0%  yes
Branch over VPN         48.9    38   120    21.8    10%  no
Congested Wi-Fi         57.4    12   140    84.3    30%  no

Utilisation of a 1 Gbps uplink from interface counters (5-minute interval):
  840 Mbps = 84% utilisation (alert threshold 70%)

Availability 'nines' -> allowed downtime per year:
       99%    87.60 hours  (5,256.0 minutes)
     99.9%     8.76 hours  (525.6 minutes)
    99.99%     0.88 hours  (52.6 minutes)
   99.999%     0.09 hours  (5.3 minutes)
